# Lesson 2, Part 3

Let's build some tools for computing integrals and charting them. 
We'll start by defining a simple plotting function and displaying axes on the standard 400 x 400 screen.

In [ ]:
%pip install ipycc
from ipycc.turtle import Turtle, showscreen, clearscreen

def plot_xy(x,y,pen_color='black'):
    t = Turtle()
    t.color(pen_color)
    t.speed('fastest')
    t.penup()
    t.goto(x[0],y[0])
    t.pendown()
    for i in range(1,len(x)):
        t.goto(x[i],y[i])

# plot horizontal and vertical axes
showscreen()
clearscreen()
plot_xy([-200,200],[0,0])
plot_xy([0,0],[-200,200])

Here is a simple quadratic function 
that can work with either single-number inputs or lists.

In [ ]:
def quadratic(x,a,b,c):
    if type(x)==int or type(x)==float:  # type() can also identify logical,tuple,list,dict,etc. 
        return  a * x * x + b * x + c
        # function halts after return, so we don't need "else:" before next line
    return [a * val * val + b * val + c for val in x]
    
a,b,c = 1,0,0
print('square of number 2 is',quadratic(2,a,b,c))
print('square of list [1,2,3,4] is',quadratic([1,2,3,4],a,b,c))

Let's plot a quadratic function against the axes.
We will place plot_xy and quadratic functions in the module functions.py.

In [ ]:
showscreen()
x = [i for i in range(-50,200,3)]
a,b,c = -0.03,3,100
y = quadratic(x,a,b,c)
plot_xy(x,y,'blue')

Our baseline allows *x* and *y* to range between -200 and +200, which is too large for some plots and too small for others. 
Let's rescale the display to fit, while still including the origin(0,0). The code below can be used with both *x* and *y*. 

In [ ]:
def rescale(x, width=390):   # could use width=400 but this will leave no room at edges
    #xmin = min(x,0)  # this generates error since we can't compare integer to list
    xmin = min(min(x),0)  # if all x>0, choose 0 to display origin
    xmax = max(max(x),0)  # if all x<0, choose 0 to display origin
    # solve for shift + mult*xmin = -width/2 and shift + mult*xmax = width/2 
    mult = width / (xmax - xmin)
    shift = -mult * (xmax + xmin) / 2
    x_rescaled = [shift + mult * val for val in x]
    return x_rescaled, shift, mult

# demo
x = [-50, 0, 50]
print('x =',x)
x_rescaled,shift,mult = rescale(x)
# insert \n before text to print on new line
print('for width=390, shift =',shift,', mult =',mult,', x_rescaled =',x_rescaled)
x_rescaled,shift,mult = rescale(x,800)
print('for width=800, shift =',shift,'mult =',mult,', x_rescaled =',x_rescaled)
x = [0, 50, 100]
print('\nx =',x)
x_rescaled,shift,mult = rescale(x)
print('for width=390, shift =',shift,', mult =',mult,', x_rescaled =',x_rescaled)

Here we rescale *x* and *y* series and plot them on rescaled *x*- and *y*-axes.

In [ ]:
def plot_xy_rescale(x,y,pen_color='black',display_axes=True,width=390):
    x_rescaled,shift_x,_ = rescale(x,width)    # outputs marked _ aren't sued later
    y_rescaled,shift_y,_ = rescale(y,width)
    if display_axes:
        # x=axis goes thru y=0, which is graphed at shift_y
        plot_xy((-width/2,width/2),(shift_y,shift_y),'gray') 
        # y-axis goes thru x=0, which is graphed at shift_x
        plot_xy((shift_x,shift_x),(-width/2,width/2),'gray') 
    plot_xy(x_rescaled,y_rescaled,pen_color)

#demo  
showscreen()
clearscreen()
x = [-2,-1,0,1,2,3,4,5,6]
y = [-20,1,50,-10,40,5,-20,0,40]
plot_xy_rescale(x,y,'pink')
plot_xy_rescale(x,y,'red',False)  # repeat to keep visible for longer
clearscreen()
x = [i for i in range(100)]
y = [-val*val for val in x]
plot_xy_rescale(x,y,'violet')

Let's use rescaling to display quadratic charts. Now we can change the parameters without losing visibility.

In [ ]:
showscreen()
clearscreen()
x = [i for i in range(-100,150,3)]
a,b,c = -0.1,4,500
y = quadratic(x,a,b,c)
plot_xy_rescale(x,y,'indigo')
clearscreen()
x = [i for i in range(-50,300,4)]
a,b,c = -0.1,4,500
y = quadratic(x,a,b,c)
plot_xy_rescale(x,y,'teal')
clearscreen()
x = [i for i in range(-40,50,2)]
a,b,c = 5,4,-1000
y = quadratic(x,a,b,c)
plot_xy_rescale(x,y,'orangered')

Next let's work out Riemann sums for a series (*x,y*). 
Here are three ways to list the heights we'll use for an upper sum.

In [ ]:
y = [i*i for i in range(-80,100)]

# Method 1: list left-hand element and replace with right-hand element if needed
upper1 = y[:-1]  # makes full copy provided each element of y is single value and not list or tuple
for i in range(len(y)-1):
    if upper1[i] < y[i+1]:
        upper1[i] = y[i+1]

# Method 2: form list of maxima from scratch
upper2 = []
for i in range(len(y)-1):
    upper2.append(max(y[i],y[i+1]))

# Method 3: use list comprehension
upper3 = [max(y[i],y[i+1]) for i in range(len(y)-1)]

print('Are these lists equivalent?',upper1==upper2==upper3)

The total area equals the sum of height times width for each slice, which can be done in a standard **for** loop or a **list comprehension**. 
However, the *x* are usually spaced regularly with a common *dx*, in which case only a single multiplication is neeeded.

In [ ]:
def area_calcs(x,height):
    # calculate total area via standard for loop
    area = 0
    for i in range(len(height)-1):
        area += height[i] * (x[i+1] - x[i])   
    # alternate method using list comprehension
    area1 = sum([height[i] * (x[i+1] - x[i]) for i in range(len(height)-1)])
    # simpler calculation if common dx
    dx = x[1] - x[0]
    area2 = sum(height[:-1]) * dx
    return area,area1,area2   # can return as three floats or as tuple

x = [i for i in range(-80,100)]
y = [val * val - 1000 for val in x]
upper = [max(y[i],y[i+1]) for i in range(len(y)-1)]
area,area1,area2 = area_calcs(x,y)       #  three floats
print('Areas for y:',area,area1,area2)   #  prints three floats in a row
print('Are areas for y equal?',area==area1==area2)
areas = area_calcs(x,upper)                   # areas is tuple with three elements 
print('Areas for upper:',area_calcs(x,upper))  # print with parentheses around tuple
print('Are areas for upper equal?',areas[0]==areas[1]==areas[2])

To choose between two mutually exclusive alternatives, we can use an **if...else** construction. 
The "**if** *H*:" statement, as we have seen, amounts to 
"run the indented block below if and only if *H* is true". 
The "**else:**" statement says "run the indented block below if and only if *H* is false".

In [ ]:
def test_if_else(real_number):
    if real_number>0:
        answer = 'positive'
    else: 
        answer = 'negative or zero'
    return answer

for num in (2,-3,0):    # parentheses not necessary   
    print(num,'is',test_if_else(num))

Here are two more ways to express **if...else** when the blocks are short and affect the same variable.

The first compresses "if *H* is true then *A*" and "else *B*" into single lines.
The second writes "*A* if *H* else *B*", also known as a ternary operator.

In [ ]:
def test_if_else1(real_number):
    if real_number>0: return 'positive'
    else: return 'negative or zero'

def test_if_else2(real_number):
    return ('positive' if real_number>0 else 'negative or zero')  # parentheses not needed
    
for num in (2,-3,-0):  # backslash '\' continues the code on the next line
    print('Are constructions with number',num,'equivalent?', \
           test_if_else(num)==test_if_else1(num)==test_if_else2(num))

This function uses an **if...else** construction to choose the summation method.

In [ ]:
# calculate total area with or without a common dx
def sum_area_slice(x,height,is_common_dx=True):
    if is_common_dx:     # this is the default method
        area = sum(height) * (x[1] - x[0])
    else:                # use this method if no common dx
        area = sum([height[i] * (x[i+1] - x[i]) for i in range(len(height))])
    return area

x = [i for i in range(-80,100)]
a,b,c = 1,0,0
y = [a * val * val + b * val + c for val in x]
upper = [max(y[i],y[i+1]) for i in range(len(y)-1)]
area = sum_area_slice(x,upper,True)    # can omit True since that is the default value
print(area)

Here is a function that draws and fills rectangular Riemann slices, without rescaling.


In [ ]:
def plot_and_fill_slices(x_rescaled, y_rescaled, baseline, fill_color):
    # baseline gives the screen coordinate cooresponding to y_rescaled = 0
    t = Turtle()
    t.speed("fastest")
    t.pencolor("gray")
    t.fillcolor(fill_color)
    for i in range(len(y_rescaled)):
        t.penup()
        t.goto(x_rescaled[i], baseline)
        t.pendown()
        t.begin_fill()
        t.goto(x_rescaled[i], y_rescaled[i])
        t.goto(x_rescaled[i+1], y_rescaled[i])
        t.goto(x_rescaled[i+1], baseline)
        t.end_fill()

# Demo: rescale the original curve once, then draw its upper slices
showscreen()
clearscreen()
x = list(range(-40, 100, 3))
a, b, c = -0.1, 7, 100
y = [a * val * val + b * val + c for val in x]
xx, _, _ = rescale(x)         # we don't use the outputs marked _
yy, baseline, _ = rescale(y)
upper = [max(yy[i], yy[i+1]) for i in range(len(yy)-1)]
plot_and_fill_slices(xx, upper, baseline, "cyan")

Calculate upper and lower Riemann bounds and their relative difference.

In [ ]:
def sum_area_slice(x,height,is_common_dx=True):
    if is_common_dx:
        return sum(height) * (x[1] - x[0])
    else: 
        return sum([height[i] * (x[i+1] - x[i]) for i in range(len(height))])

def calc_Riemann_bounds(x,y,is_common_dx=True):
    upper = [max(y[i],y[i+1]) for i in range(len(y)-1)]
    area_upper = sum_area_slice(x,upper,is_common_dx)
    lower = [min(y[i],y[i+1]) for i in range(len(y)-1)]
    area_lower = sum_area_slice(x,lower,is_common_dx)
    rel_diff = (area_upper - area_lower) / (0.5 * abs(area_lower + area_upper))
    print(round(area_lower,5), '< area <', round(area_upper,5) )
    print('relative difference',round(rel_diff,5),'between bounds')
               
x = list(range(-40,100,1))   # = [i for i in range(-40,100,1)]
a,b,c = -0.05,5,100
y = [a * val * val + b * val + c for val in x]
calc_Riemann_bounds(x,y)

To display upper slices, lower slices, and the curve on one chart, we first rescale the original *x* and *y*. We then calculate the upper and lower heights of the rescaled *y* and use those plot the Riemann slices.

In [ ]:
def plot_Riemann_areas(x, y):
    xx, _, _ = rescale(x)            # outputs marked _ are not used elswhere in code
    yy, baseline, _ = rescale(y)
    upper = [max(yy[i], yy[i+1]) for i in range(len(yy)-1)]
    lower = [min(yy[i], yy[i+1]) for i in range(len(yy)-1)]
    plot_and_fill_slices(xx, upper, baseline, "cyan")
    plot_and_fill_slices(xx, lower, baseline, "lightpink")
    plot_xy(xx, yy, "black")

# Demo: all three drawings now share exactly the same scale
showscreen()
clearscreen()
x = list(range(-40, 100, 3))
a, b, c = -0.1, 7, 100
y = [a * val * val + b * val + c for val in x]
plot_Riemann_areas(x, y)

There is no need to restrict our calculations to quadratics. Here is a general formula for cubics, for which we can calculate Riemann bounds.

In [ ]:
def cubic(x,a,b,c,d):   # works with int/float x or sequences
    if type(x)==int or type(x)==float:  
        return (a*x*x + b*x + c) * x + d
    return [(a*val*val + b*val + c) * val + d for val in x]

# demo
showscreen()
clearscreen()
x = list(range(-10,80))  # same as [i for i in range(-10,80)]
a,b,c,d = -0.05,5,-50,-1500
y = cubic(x,a,b,c,d)
calc_Riemann_bounds(x,y)
plot_Riemann_areas_new(x,y)